## 爬虫

In [1]:
import requests as req
import re
url='http://wenshu.court.gov.cn/List/ListContent?'
my_headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/56.0.2924.87 Safari/537.36',
                'Referer':'http://wenshu.court.gov.cn/List/List?sorttype=1&conditions=searchWord+2+AJLX++%E6%A1%88%E4%BB%B6%E7%B1%BB%E5%9E%8B:%E6%B0%91%E4%BA%8B%E6%A1%88%E4%BB%B6'
               }
data={'Param':'案件类型:民事案件', 'Index': 2,'Page':'5','Order':'法院层级','Direction':'asc'}
r=req.post(url,headers=my_headers,data=data)
raw=r.json()
pattern1= re.compile('"裁判日期":"(.*?)"',re.S)
date= re.findall(pattern1,raw)
pattern2= re.compile('"案号":"(.*?)"',re.S)
num= re.findall(pattern2,raw)
pattern3= re.compile('"案件名称":"(.*?)"',re.S)
title= re.findall(pattern3,raw)
pattern4= re.compile('"裁判要旨段原文":"(.*?)"',re.S)
content= re.findall(pattern4,raw)
content

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## 数据筛选并转json

In [15]:
import pandas as pd 
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2.csv')
length_series = df['全文'].str.len()

In [16]:
df = df.drop(columns=['index'])

In [17]:
df=df.reset_index()
df.head(1)

,index,原始链接,案号,案件名称,法院,所属地区,案件类型,案件类型编码,来源,审理程序,裁判日期,公开日期,当事人,案由,法律依据,全文
0,0,https://wenshu.court.gov.cn/website/wenshu/181...,（2021）湘1124刑初37号,何茶开、胡良波走私、贩卖、运输、制造毒品、容留他人吸毒一审刑事判决书,湖南省道县人民法院,湖南省道县,刑事案件,4,www.macrodatas.cn,刑事一审,2021-02-01,2021-02-02,何茶开；胡良波；许兰喜；朱军彩；周得艳；文德胜；王景鑫；田治令,走私、贩卖、运输、制造毒品；容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,湖南省道县人民法院 刑 事 判 决 书 （2021）湘1124刑初37号 公诉机关湖南省道...


In [19]:
df.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2.csv',index=False)

In [43]:
length_distribution = length_series.value_counts()
print(length_distribution)

740.0      24
797.0      20
790.0      20
788.0      20
758.0      19
           ..
3765.0      1
3639.0      1
357.0       1
14972.0     1
575.0       1
Name: 全文, Length: 2460, dtype: int64


In [44]:
length_series.hist(bins=range(length_series.min(), length_series.max() + 2))

TypeError: 'float' object cannot be interpreted as an integer

In [20]:
import re

def extract_party_info(text):
    # Pattern to extract the defendant's information
    party_info_pattern = r"被告人(.*?)(，.*?，.*?，.*?出生)"
    
    # Search for the basic information
    match = re.search(party_info_pattern, text, re.S)
    if not match:
        return None  # 如果匹配不到，直接返回 None
    
    party_info = match.group(0)
    
    # Extract specific fields using regex and check for None before accessing group
    name_match = re.search(r"被告人(.*?)[，。]", party_info)
    name = name_match.group(1).strip() if name_match else ""
    
    gender_match = re.search(r"，(男|女)[，。]", party_info)
    gender = gender_match.group(1).strip() if gender_match else ""
    
    ethnicity_match = re.search(r"，(汉族|满族|蒙古族|壮族)[，。]", party_info)
    ethnicity = ethnicity_match.group(1).strip() if ethnicity_match else ""
    
    birth_date_match = re.search(r"(\d{4}年\d{1,2}月\d{1,2}日)出生", party_info)
    birth_date = birth_date_match.group(1).strip() if birth_date_match else ""
    
    # Extract birthplace if it exists
    birthplace_match = re.search(r"出生于(.*?省.*?市)", text)
    birthplace = birthplace_match.group(1).strip() if birthplace_match else ""

    return {
        "姓名": name,
        "性别": gender,
        "民族": ethnicity,
        "出生年月": birth_date,
        "户籍地": birthplace  # No explicit field in the text
    }


def extract_judgment_info(text):
    # 提取公诉机关
    prosecution_match = re.search(r"公诉机关(.*?人民检察院)", text)
    prosecution = prosecution_match.group(1).strip() if prosecution_match else ""

    # 提取结尾的审判员
    judge_match = re.search(r"审判员\s+(\S+)$", text, re.M)
    judge = judge_match.group(1).strip() if judge_match else ""
    # 提取审判长
    presiding_judge_match = re.search(r"审\s*判\s*长\s+(\S+)", text)
    presiding_judge = presiding_judge_match.group(1).strip() if presiding_judge_match else ""

    # 提取书记员
    clerk_match = re.search(r"书\s*记\s*员\s+(\S+)", text)
    clerk = clerk_match.group(1).strip() if clerk_match else ""
    return {
        "公诉机关": prosecution,
        "审判员": judge,
        "审判长": presiding_judge,
        "书记员": clerk
    }

def extract_main_body(text):
    # 正则表达式匹配正文的开始和结束
    main_body_pattern = r"(指控被告人.*?犯.*?罪)(.*?如不服本判决)"
    match = re.search(main_body_pattern, text, re.S)
    
    if match:
        # 提取正文部分
        main_body = match.group(0).strip()
        return main_body
    else:
        # 如果没有找到正文，返回空字符串
        return ""

def assemble_to_json(text):
    # 调用各个提取函数
    party_info = extract_party_info(text)
    judgment_info = extract_judgment_info(text)
    main_body = extract_main_body(text)
    
    # 构建 JSON 格式
    result = {
        "首部": {
            "公诉机关": judgment_info.get("公诉机关", "")
        },
        "当事人基本情况": party_info if party_info else {},
        "正文部分": {
            "文本": main_body
        },
        "落款": {
            "审判长": judgment_info.get("审判长", ""),
            "审判员": judgment_info.get("审判员", ""),
            "书记员": judgment_info.get("书记员", "")
        }
    }

    return [result]    
# Example input text
judgment_text = """
  四川省金阳县人民法院 刑 事 判 决 书 （2021）川3430刑初5号 公诉机关四川省金阳县人民检察院。 被告人勒古木尔作，女，现年36岁（1984年5月6日出生），彝族，文盲，村民，户籍所在地四川省金阳县，捕前住四川省金阳县，因犯贩卖毒品罪、容留他人吸毒罪，于2016年3月27日被莆田市公安局荔城分局刑事拘留（3月26日被抓获），经莆田市公安局荔城分局决定，于2016年4月23日被取保候审。该案于2019年8月8日移送四川省凉山州金阳县公安局管辖，金阳县。经四川省金阳县人民法院决定，于2021年1月18日被金阳县公安局执行逮捕。现羁押于金阳县看守所。 四川省金阳县人民检察院以金检一部刑诉［2020］Z48号起诉书指控被告人勒古木尔作犯贩卖毒品罪、容留他人吸毒罪，于2021年1月5日向本院提起公诉，本院依法组成合议庭，公开开庭审理了本案。金阳县人民检察院指派检察员白史则出庭支持公诉。翻译人员此么春立，被告人勒古木尔作到庭参加诉讼，现已审理终结。 四川省金阳县人民检察院指控：2016年3月初至3月26日期间，苏某（另案处理）勒古木尔作共同出资购买毒品海洛因5克，之后分别于2016年3月8日、3月25日、3月26日在福建省莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内（房东余某）三次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员哑某并容留哑某在房间内吸食。苏某、勒古木尔作分别于2016年3月10日左右、3月26日在莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内两次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员黄某并容留黄某在房间内当场吸食。苏某、勒古木尔作分别于2016年3月24日、3月25日、3月26日在莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内三次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员米某并容留米某在房间内吸食。苏某、勒古木尔作分别于2016年3月中旬某日以及两天后某日两次在莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内以0.06克／50元人民币价格贩卖0.06克毒品海洛因给张某并容留张某当场吸食。2016年3月26日18时许，在莆田市荔城区&times;&times;镇&times;&times;村苏某、勒古木尔作居住的出租屋内，苏某、勒古木尔作向哑某、黄某、米某、张某等人贩卖毒品海洛因并容留哑某、黄某、米某、张某等人在房间内吸食时被民警查获。2016年3月27日7时许，民警在现场检查时，在现场扣押苏某、勒古木尔作持有疑似毒品海洛因的块状物4.0克。经莆田市公安局物证鉴定所鉴定，从苏某、勒古木尔作持有的疑似毒品海洛因的块状物送检的检材物中检出海洛因成分。 针对上述指控，公诉机关当庭出示了物证毒品（照片呈现）、常住人口信息、违反犯罪记录表、在逃人员信息登记表、现场检查报告书、证人张某、米某、黄某等人的证言、被告人勒古木尔作的供述、鉴定意见书、检查笔录、辨认笔录、扣押笔录、审讯视频光盘等证据。 公诉机关认为，被告人勒古木尔作以营利为目的，非法贩卖毒品海洛因并容留他人吸毒，其行为已构成贩卖毒品罪、容留他人吸毒罪，建议对被告勒古木尔作贩卖毒品罪，判处有期徒刑三年以上四年以下，并处罚金；犯容留他人吸毒罪，判处有期徒刑一年以下，并处罚金；数罪并罚，在有期徒刑四年量刑，并处罚金。 被告人勒古木尔作以自己是购买毒品吸食的，是苏呷木尔火一个人贩卖毒品，自己什么都没有干，是无罪的理由进行辩护。 经审理查明：2016年3月初至3月26日期间，被告人苏某（已判刑）与被告人勒古木尔作共同出资购买毒品海洛因5克，之后分别于2016年3月8日、3月25日、3月26日在福建省莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内（房东余某）三次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员哑某并容留哑某在房间内吸食。 被告人苏某、被告人勒古木尔作分别于2016年3月10日左右、3月26日在莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内两次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员黄某并容留黄某在房间内当场吸食。 被告人苏某、被告人勒古木尔作分别于2016年3月24日、3月25日、3月26日在莆田市荔城区&times;&times;镇&times;&times;村&times;&times;人居住的出租屋内三次共同以0.1克／50.00元人民币的价格贩卖0.1克毒品海洛因给吸毒人员米某并容留米某在房间内吸食。 被告人苏某、被告人勒古木尔作分别于2016年3月中旬某日以及两天后某日两次在莆田市荔城区&times;&times;镇&times;&times;村&times;&tim
"""

# Output the result
print(assemble_to_json(judgment_text))

[{'首部': {'公诉机关': '四川省金阳县人民检察院'}, '当事人基本情况': {}, '正文部分': {'文本': ''}, '落款': {'审判长': '', '审判员': '', '书记员': ''}}]


In [21]:
import pandas as pd
import json
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, TimeoutError


def process_csv_to_json(input_csv_path, output_json_path, start_row=0, timeout=5):
    # 读取 CSV 文件
    df = pd.read_csv(input_csv_path)

    # 确保有“全文”列
    
    if "全文" not in df.columns:
        raise ValueError("输入的 CSV 文件必须包含 '全文' 列")

    # 打开 JSON 文件，以追加模式写入数据
    with open(output_json_path, "w", encoding="utf-8") as json_file:
        json_file.write("[\n")  # 写入 JSON 文件的开始部分

        # 从指定行开始遍历
        for index, row in tqdm(df.iloc[start_row:].iterrows(), 
                               total=len(df) - start_row, 
                               desc='Processing', 
                               unit='rows', 
                               dynamic_ncols=True):
            text = row["全文"]
            if (len(text)>3500):
                continue
            try:
                # 使用 ThreadPoolExecutor 来设置超时
                with ThreadPoolExecutor(max_workers=1) as executor:
                    future = executor.submit(assemble_to_json, text)
                    processed_data = future.result(timeout=timeout)

                for item in processed_data:
                    item["index"] = row["index"]
                    
                # 写入处理后的 JSON 数据
                for i, item in enumerate(processed_data):
                    json.dump(item, json_file, ensure_ascii=False, indent=4)
                    # 仅在最后一行不加逗号
                    if index != len(df) - 1 or i != len(processed_data) - 1:
                        json_file.write(",\n")
            except TimeoutError:
                print(f"处理第 {index} 行超时，跳过该行。")
            except Exception as e:
                print(f"处理第 {index} 行时出错，跳过该行。错误信息: {e}")

        json_file.write("\n]")  # 写入 JSON 文件的结束部分

    print(f"处理完成，结果已保存到 {output_json_path}")

# 示例用法
input_csv = "/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2.csv"  # 替换为实际的 CSV 文件路径
output_json = "/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2JSON.json"  # 生成的 JSON 文件路径
start_row = 0  # 从第 845 行开始处理

process_csv_to_json(input_csv, output_json, start_row=start_row)

Processing: 100%|██████████| 6281/6281 [51:28<00:00,  2.03rows/s]  


处理完成，结果已保存到 /Users/yuu/Downloads/毕业设计/spider_bysj/毒品2JSON.json


In [22]:
import json

# 读取 JSON 文件
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2JSON.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

# 定义过滤逻辑
filtered_data = []
for item in data:
    # 确保有'正文部分'和'当事人基本情况'键
    if '正文部分' in item and '当事人基本情况' in item:
        # 获取必要的字段
        text = item['正文部分'].get('文本', '').strip()
        basic_info = item['当事人基本情况']
        # 检查文本和当事人基本情况的必要字段是否不为空
        if text and all(basic_info.get(key, '').strip() for key in ['姓名', '性别', '民族', '出生年月']):
            filtered_data.append(item)

# 输出或保存过滤后的数据
with open('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2filtered.json', 'w', encoding='utf-8') as file:
    json.dump(filtered_data, file, ensure_ascii=False, indent=4)

print(f"过滤完成，共保留 {len(filtered_data)} 条数据。")

过滤完成，共保留 182 条数据。


### 数据标注

In [23]:
import json

def clean_text(text):
    """
    删除文本中的 &time; 字符。
    """
    return text.replace("&times;&times;", "")

def process_json(input_json_file, output_json_file):
    # 打开 JSON 文件
    with open(input_json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 遍历数据并清理 "正文部分" 中的 "文本"
    for record in data:
        if "正文部分" in record and "文本" in record["正文部分"]:
            original_text = record["正文部分"]["文本"]
            cleaned_text = clean_text(original_text)
            record["正文部分"]["文本"] = cleaned_text
        if "当事人基本情况" in record and "户籍地" in record["当事人基本情况"]:
            original_text = record["当事人基本情况"]["户籍地"]
            cleaned_text = clean_text(original_text)
            record["当事人基本情况"]["户籍地"] = cleaned_text
    
    # 将结果保存到输出文件
    with open(output_json_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    print(f"处理完成！结果已保存到 {output_json_file}")

# 使用示例
input_json_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2filtered.json"  # 输入 JSON 文件路径
output_json_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json"  # 输出清理后的 JSON 文件路径

process_json(input_json_file, output_json_file)

处理完成！结果已保存到 /Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json


In [18]:
import json
import random

def split_text(text, min_len=100, max_len=200):
    """
    将文本分割为指定长度的段落，长度在 min_len 和 max_len 之间随机生成。
    """
    chunks = []
    i = 0
    while i < len(text):
        segment_length = random.randint(min_len, max_len)
        chunks.append(text[i:i + segment_length])
        i += segment_length
    return chunks

def process_json_to_txt(input_json_file, output_txt_file):
    # 打开 JSON 文件
    with open(input_json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 准备保存的内容
    result_lines = []
    
    for record in data:
        # 提取 "正文部分" 中的 "文本"
        text = record.get("正文部分", {}).get("文本", "")
        if text:
            # 按段分割文本
            split_chunks = split_text(text)
            result_lines.extend(split_chunks)
    
    # 打乱所有数据
    random.shuffle(result_lines)
    
    # 将结果保存到 TXT 文件
    with open(output_txt_file, 'w', encoding='utf-8') as f:
        for line in result_lines:
            f.write(line + "\n")
    
    print(f"处理完成！结果已保存到 {output_txt_file}")

# 使用示例
input_json_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品clean.json"  # 输入 JSON 文件路径
output_txt_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/drug.txt"  # 输出 TXT 文件路径

process_json_to_txt(input_json_file, output_txt_file)

处理完成！结果已保存到 /Users/yuu/Downloads/毕业设计/spider_bysj/tmp/drug.txt


In [24]:
import json
import re

def extract_penalty(text):
    """
    从文本中提取判罚内容，包括刑期和罚金。
    """
    penalty = {"刑期": "", "罚金": ""}
    
    # 匹配刑期，如“有期徒刑一年五个月”、“拘役六个月”或“有期徒刑六个月”
    prison_pattern = r"(有期徒刑[一二三四五六七八九十零○百千\d]+年[一二三四五六七八九十零○百千\d]*个月*|有期徒刑[一二三四五六七八九十零○百千\d]+个月*|拘役[一二三四五六七八九十零○百千\d]+年[一二三四五六七八九十零○百千\d]*个月*|拘役[一二三四五六七八九十零○百千\d]+个月*)"
    prison_match = re.search(prison_pattern, text)
    if prison_match:
        penalty["刑期"] = prison_match.group()
    
    # 匹配罚金，如“罚金人民币八千元”
    fine_pattern = r"罚金人民币[一二三四五六七八九十百千万亿零○\d]+元"
    fine_match = re.search(fine_pattern, text)
    if fine_match:
        penalty["罚金"] = fine_match.group()
    
    return penalty

def process_json(input_file, output_file):
    """
    从输入 JSON 文件提取判罚内容，并将其写入到新的 JSON 文件中。
    """
    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    for record in data:
        if "正文部分" in record and "文本" in record["正文部分"]:
            text = record["正文部分"]["文本"]
            # 提取判罚内容
            penalty = extract_penalty(text)
            # 写入 "判罚" 部分
            record["判罚"] = penalty
    
    # 保存到输出文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)
    
    print(f"处理完成！结果已保存到 {output_file}")
# 使用示例
input_json_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json"
output_json_file = "/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json"  # 输出 JSON 文件路径

process_json(input_json_file, output_json_file)

处理完成！结果已保存到 /Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json


In [25]:
file_path='/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2clean.json'
with open(file_path, 'r', encoding='utf-8') as file:
        data = json.load(file) 

import pandas as pd 
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/毒品2.csv')
df1 = pd.DataFrame(columns=['index', '案号', '案件名称','法院','所属地区','审理程序','裁判日期','当事人','案由','法律依据'])
index_list=[]
for record in data:
        index_list.append(record['index'])
df_filtered = df.loc[index_list]  


In [26]:
df_filtered=df_filtered[['index', '案号', '案件名称','法院','所属地区','审理程序','裁判日期','当事人','案由','法律依据']]
df_filtered.columns

Index(['index', '案号', '案件名称', '法院', '所属地区', '审理程序', '裁判日期', '当事人', '案由',
       '法律依据'],
      dtype='object')

In [27]:
df_filtered

,index,案号,案件名称,法院,所属地区,审理程序,裁判日期,当事人,案由,法律依据
119,119,（2021）浙0211刑初45号,马飞、徐立本走私、贩卖、运输、制造毒品罪一审刑事判决书,宁波市镇海区人民法院,宁波市,刑事一审,2021-02-04,马飞；徐立本,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...
283,283,（2021）鲁0303刑初47号,宋晨、朱晓冬走私、贩卖、运输、制造毒品、容留他人吸毒一审刑事判决书,山东省淄博市张店区人民法院,山东省淄博市,刑事一审,2021-02-02,宋晨；朱晓冬,走私、贩卖、运输、制造毒品；容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...
443,443,（2021）鲁0523刑初14号,郑学峰容留他人吸毒一审刑事判决书,山东省广饶县人民法院,山东省广饶县,刑事一审,2021-02-08,郑学峰,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...
462,462,（2021）黔0304刑初12号,罗昭俊容留他人吸毒一审刑事判决书,贵州省遵义市播州区人民法院,贵州省遵义市,刑事一审,2021-02-01,罗昭俊,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...
557,557,（2021）鄂0102刑初115号,朱月玲、黄昭容留他人吸毒一审刑事判决书,湖北省武汉市江岸区人民法院,湖北省武汉市,刑事一审,2021-02-01,朱月玲；黄昭,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十八条；《中华人民共和国刑法（1997年）》:...
...,...,...,...,...,...,...,...,...,...,...
6144,6144,（2021）桂0702刑初54号,梁焕彬、吴志杰走私、贩卖、运输、制造毒品罪一审刑事判决书,钦州市钦南区人民法院,钦州市,刑事一审,2021-02-08,梁焕彬；吴志杰,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...
6226,6226,（2021）鲁0203刑初110号,丁一冰、辛晓鹏走私、贩卖、运输、制造毒品一审刑事判决书,山东省青岛市市北区人民法院,山东省青岛市,刑事一审,2021-02-25,丁一冰；辛晓鹏,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...
6260,6260,（2021）川0182刑初48号,翁梦玲、冯诚诚走私、贩卖、运输、制造毒品罪一审刑事判决书,彭州市人民法院,彭州市,刑事一审,2021-02-05,翁梦玲；冯诚诚,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...
6266,6266,（2021）皖1621刑初33号,秦永刚容留他人吸毒罪一审刑事判决书,涡阳县人民法院,涡阳县,刑事一审,2021-02-18,秦永刚,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...


In [28]:
df_json = pd.json_normalize(data)
df_json

,index,首部.公诉机关,当事人基本情况.姓名,当事人基本情况.性别,当事人基本情况.民族,当事人基本情况.出生年月,当事人基本情况.户籍地,正文部分.文本,落款.审判长,落款.审判员,落款.书记员,判罚.刑期,判罚.罚金
0,119,浙江省宁波市镇海区人民检察院,马飞,男,汉族,1990年12月8日,河南省平舆县，汉族，小学文化，务工，户籍所在地河南省驻马店市,指控被告人马飞、徐立本犯贩卖毒品罪，于2021年1月18日向本院提起公诉。因本院对该案没有管...,,,沈国芳,有期徒刑一年二个月,罚金人民币二千元
1,283,淄博市张店区人民检察院,宋晨,男,汉族,1980年3月20日,黑龙江省牡丹江市,指控被告人宋晨犯贩卖毒品罪、被告人朱晓冬犯容留他人吸毒罪，于2021年1月18日向本院提起公...,刘华蕾,,李,有期徒刑六个月,罚金人民币五千元
2,443,山东省广饶县人民检察院,郑学峰,男,汉族,1990年12月5日,山东省阳信县，高中文化，农民，户籍所在地广饶县。因涉嫌犯容留他人吸毒罪，2020年11月16...,指控被告人郑学峰犯容留他人吸毒罪向本院提起公诉，本院于2021年1月18日立案，依法组成合议...,徐,,韩,有期徒刑九个月,罚金人民币三千元
3,462,遵义市播州区人民检察院,罗昭俊,男,汉族,1969年6月16日,贵州省遵义市,指控被告人罗昭俊犯容留他人吸毒罪，向本院提起公诉。本院于2021年1月15日受理后，依法适用...,,,,有期徒刑七个月,罚金人民币一千元
4,557,武汉市江岸区人民检察院,朱月玲,女,汉族,1973年7月9日,湖北省云梦县，汉族，小学文化，无职业，户籍地武汉市,指控被告人朱月玲、黄昭犯容留他人吸毒罪，于2021年1月20日向本院提起公诉，本院依法适用简...,,,,拘役五个月,罚金人民币一千元
...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,6144,广西壮族自治区钦州市钦南区人民检察院,梁焕彬,男,汉族,1997年9月5日,,指控被告人梁焕彬、吴志杰犯贩卖毒品罪，于2021年1月23日向本院提起公诉，本院受理后依法适...,,,黄文亮,有期徒刑九个月,
178,6226,山东省青岛市市北区人民检察院,丁一冰,男,汉族,1990年8月30日,,指控被告人丁一冰、辛晓鹏犯贩卖毒品罪，于2021年2月5日向本院提起公诉。本院受理后，依法组...,范家强人民陪审员,,陈,有期徒刑六个月,罚金人民币三千元
179,6260,四川省彭州市人民检察院,翁梦玲,女,汉族,1992年2月6日,重庆市，汉族，中专文化，无业，户籍所在地四川省成都市,指控被告人翁梦玲、冯诚诚犯贩卖毒品罪，于2021年1月12日向本院提起公诉。本院依法适用简易...,黄基伟,,高,有期徒刑七个月,罚金人民币三千元
180,6266,涡阳县人民检察院,秦永刚,男,汉族,1987年4月6日,,指控被告人秦永刚犯容留他人吸毒罪，于2021年1月19日向本院提起公诉。本院依法组成合议庭，...,赵修珠,,徐,有期徒刑七个月,


In [29]:
df_json.columns

Index(['index', '首部.公诉机关', '当事人基本情况.姓名', '当事人基本情况.性别', '当事人基本情况.民族',
       '当事人基本情况.出生年月', '当事人基本情况.户籍地', '正文部分.文本', '落款.审判长', '落款.审判员', '落款.书记员',
       '判罚.刑期', '判罚.罚金'],
      dtype='object')

In [30]:
df_json=df_json[['index', '首部.公诉机关', '当事人基本情况.姓名', '当事人基本情况.性别', '当事人基本情况.民族',
       '当事人基本情况.出生年月',  '正文部分.文本', '判罚.刑期', '判罚.罚金']]
df_json.columns

Index(['index', '首部.公诉机关', '当事人基本情况.姓名', '当事人基本情况.性别', '当事人基本情况.民族',
       '当事人基本情况.出生年月', '正文部分.文本', '判罚.刑期', '判罚.罚金'],
      dtype='object')

In [31]:
df_merged = pd.merge(df_filtered, df_json, on='index', how='left')

# 重命名列以去掉 "正文部分." 前缀（可选）
df_merged.rename(columns={"正文部分.文本": "文本","首部.公诉机关":"公诉机关","当事人基本情况.姓名":"姓名","当事人基本情况.姓别":"姓别","当事人基本情况.民族":"民族","当事人基本情况.出生年月":"出生年月","判罚.刑期":"刑期","判罚.罚金":"罚金"}, inplace=True)
df_merged

,index,案号,案件名称,法院,所属地区,审理程序,裁判日期,当事人,案由,法律依据,公诉机关,姓名,当事人基本情况.性别,民族,出生年月,文本,刑期,罚金
0,119,（2021）浙0211刑初45号,马飞、徐立本走私、贩卖、运输、制造毒品罪一审刑事判决书,宁波市镇海区人民法院,宁波市,刑事一审,2021-02-04,马飞；徐立本,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,浙江省宁波市镇海区人民检察院,马飞,男,汉族,1990年12月8日,指控被告人马飞、徐立本犯贩卖毒品罪，于2021年1月18日向本院提起公诉。因本院对该案没有管...,有期徒刑一年二个月,罚金人民币二千元
1,283,（2021）鲁0303刑初47号,宋晨、朱晓冬走私、贩卖、运输、制造毒品、容留他人吸毒一审刑事判决书,山东省淄博市张店区人民法院,山东省淄博市,刑事一审,2021-02-02,宋晨；朱晓冬,走私、贩卖、运输、制造毒品；容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,淄博市张店区人民检察院,宋晨,男,汉族,1980年3月20日,指控被告人宋晨犯贩卖毒品罪、被告人朱晓冬犯容留他人吸毒罪，于2021年1月18日向本院提起公...,有期徒刑六个月,罚金人民币五千元
2,443,（2021）鲁0523刑初14号,郑学峰容留他人吸毒一审刑事判决书,山东省广饶县人民法院,山东省广饶县,刑事一审,2021-02-08,郑学峰,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...,山东省广饶县人民检察院,郑学峰,男,汉族,1990年12月5日,指控被告人郑学峰犯容留他人吸毒罪向本院提起公诉，本院于2021年1月18日立案，依法组成合议...,有期徒刑九个月,罚金人民币三千元
3,462,（2021）黔0304刑初12号,罗昭俊容留他人吸毒一审刑事判决书,贵州省遵义市播州区人民法院,贵州省遵义市,刑事一审,2021-02-01,罗昭俊,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...,遵义市播州区人民检察院,罗昭俊,男,汉族,1969年6月16日,指控被告人罗昭俊犯容留他人吸毒罪，向本院提起公诉。本院于2021年1月15日受理后，依法适用...,有期徒刑七个月,罚金人民币一千元
4,557,（2021）鄂0102刑初115号,朱月玲、黄昭容留他人吸毒一审刑事判决书,湖北省武汉市江岸区人民法院,湖北省武汉市,刑事一审,2021-02-01,朱月玲；黄昭,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十八条；《中华人民共和国刑法（1997年）》:...,武汉市江岸区人民检察院,朱月玲,女,汉族,1973年7月9日,指控被告人朱月玲、黄昭犯容留他人吸毒罪，于2021年1月20日向本院提起公诉，本院依法适用简...,拘役五个月,罚金人民币一千元
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177,6144,（2021）桂0702刑初54号,梁焕彬、吴志杰走私、贩卖、运输、制造毒品罪一审刑事判决书,钦州市钦南区人民法院,钦州市,刑事一审,2021-02-08,梁焕彬；吴志杰,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,广西壮族自治区钦州市钦南区人民检察院,梁焕彬,男,汉族,1997年9月5日,指控被告人梁焕彬、吴志杰犯贩卖毒品罪，于2021年1月23日向本院提起公诉，本院受理后依法适...,有期徒刑九个月,
178,6226,（2021）鲁0203刑初110号,丁一冰、辛晓鹏走私、贩卖、运输、制造毒品一审刑事判决书,山东省青岛市市北区人民法院,山东省青岛市,刑事一审,2021-02-25,丁一冰；辛晓鹏,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,山东省青岛市市北区人民检察院,丁一冰,男,汉族,1990年8月30日,指控被告人丁一冰、辛晓鹏犯贩卖毒品罪，于2021年2月5日向本院提起公诉。本院受理后，依法组...,有期徒刑六个月,罚金人民币三千元
179,6260,（2021）川0182刑初48号,翁梦玲、冯诚诚走私、贩卖、运输、制造毒品罪一审刑事判决书,彭州市人民法院,彭州市,刑事一审,2021-02-05,翁梦玲；冯诚诚,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,四川省彭州市人民检察院,翁梦玲,女,汉族,1992年2月6日,指控被告人翁梦玲、冯诚诚犯贩卖毒品罪，于2021年1月12日向本院提起公诉。本院依法适用简易...,有期徒刑七个月,罚金人民币三千元
180,6266,（2021）皖1621刑初33号,秦永刚容留他人吸毒罪一审刑事判决书,涡阳县人民法院,涡阳县,刑事一审,2021-02-18,秦永刚,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...,涡阳县人民检察院,秦永刚,男,汉族,1987年4月6日,指控被告人秦永刚犯容留他人吸毒罪，于2021年1月19日向本院提起公诉。本院依法组成合议庭，...,有期徒刑七个月,


In [32]:
df_merged.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2_clean.csv',index=False)

In [33]:
import pandas as pd 
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2_clean.csv')
df['罪名'] = df['案由'].astype(str) + '罪'
df['罪名']

0             走私、贩卖、运输、制造毒品罪
1      走私、贩卖、运输、制造毒品；容留他人吸毒罪
2                    容留他人吸毒罪
3                    容留他人吸毒罪
4                    容留他人吸毒罪
               ...          
177           走私、贩卖、运输、制造毒品罪
178           走私、贩卖、运输、制造毒品罪
179           走私、贩卖、运输、制造毒品罪
180                  容留他人吸毒罪
181           走私、贩卖、运输、制造毒品罪
Name: 罪名, Length: 182, dtype: object

In [34]:
df.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2_clean.csv',index=False)

In [35]:
df1=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2_clean.csv')
df2=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品1_clean.csv')
result = pd.concat([df1, df2], axis=0, ignore_index=True)
result

,index,案号,案件名称,法院,所属地区,审理程序,裁判日期,当事人,案由,法律依据,公诉机关,姓名,当事人基本情况.性别,民族,出生年月,文本,刑期,罚金,罪名
0,119,（2021）浙0211刑初45号,马飞、徐立本走私、贩卖、运输、制造毒品罪一审刑事判决书,宁波市镇海区人民法院,宁波市,刑事一审,2021-02-04,马飞；徐立本,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,浙江省宁波市镇海区人民检察院,马飞,男,汉族,1990年12月8日,指控被告人马飞、徐立本犯贩卖毒品罪，于2021年1月18日向本院提起公诉。因本院对该案没有管...,有期徒刑一年二个月,罚金人民币二千元,走私、贩卖、运输、制造毒品罪
1,283,（2021）鲁0303刑初47号,宋晨、朱晓冬走私、贩卖、运输、制造毒品、容留他人吸毒一审刑事判决书,山东省淄博市张店区人民法院,山东省淄博市,刑事一审,2021-02-02,宋晨；朱晓冬,走私、贩卖、运输、制造毒品；容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,淄博市张店区人民检察院,宋晨,男,汉族,1980年3月20日,指控被告人宋晨犯贩卖毒品罪、被告人朱晓冬犯容留他人吸毒罪，于2021年1月18日向本院提起公...,有期徒刑六个月,罚金人民币五千元,走私、贩卖、运输、制造毒品；容留他人吸毒罪
2,443,（2021）鲁0523刑初14号,郑学峰容留他人吸毒一审刑事判决书,山东省广饶县人民法院,山东省广饶县,刑事一审,2021-02-08,郑学峰,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...,山东省广饶县人民检察院,郑学峰,男,汉族,1990年12月5日,指控被告人郑学峰犯容留他人吸毒罪向本院提起公诉，本院于2021年1月18日立案，依法组成合议...,有期徒刑九个月,罚金人民币三千元,容留他人吸毒罪
3,462,（2021）黔0304刑初12号,罗昭俊容留他人吸毒一审刑事判决书,贵州省遵义市播州区人民法院,贵州省遵义市,刑事一审,2021-02-01,罗昭俊,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百五十四条；《中华人民共和国刑法（1997年）》:...,遵义市播州区人民检察院,罗昭俊,男,汉族,1969年6月16日,指控被告人罗昭俊犯容留他人吸毒罪，向本院提起公诉。本院于2021年1月15日受理后，依法适用...,有期徒刑七个月,罚金人民币一千元,容留他人吸毒罪
4,557,（2021）鄂0102刑初115号,朱月玲、黄昭容留他人吸毒一审刑事判决书,湖北省武汉市江岸区人民法院,湖北省武汉市,刑事一审,2021-02-01,朱月玲；黄昭,容留他人吸毒,《中华人民共和国刑法（1997年）》:第三百四十八条；《中华人民共和国刑法（1997年）》:...,武汉市江岸区人民检察院,朱月玲,女,汉族,1973年7月9日,指控被告人朱月玲、黄昭犯容留他人吸毒罪，于2021年1月20日向本院提起公诉，本院依法适用简...,拘役五个月,罚金人民币一千元,容留他人吸毒罪
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,7063,（2021）粤0104刑初24号,杜会宁一审刑事判决书,广东省广州市越秀区人民法院,广东省广州市,刑事一审,2021-01-08,杜会宁,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,广东省广州市越秀区人民检察院,杜会宁,男,汉族,1974年6月5日,指控被告人杜会宁犯贩卖毒品罪。本院适用刑事案件速裁程序，实行独任审判，公开开庭审理了本案。公...,有期徒刑八个月,NaN,走私、贩卖、运输、制造毒品罪
424,7065,（2021）粤0104刑初25号,胡秋良一审刑事判决书,广东省广州市越秀区人民法院,广东省广州市,刑事一审,2021-01-08,胡秋良,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,广东省广州市越秀区人民检察院,胡秋良,男,汉族,1958年6月18日,指控被告人胡秋良犯贩卖毒品罪。本院适用刑事案件速裁程序，实行独任审判，公开开庭审理了本案。公...,有期徒刑六个月,NaN,走私、贩卖、运输、制造毒品罪
425,7074,（2020）粤1972刑初4244号,欧阳科军、向武强走私、贩卖、运输、制造毒品罪一案刑事一审判决书,东莞市第二人民法院,东莞市,刑事一审,2021-01-28,欧阳科军；向武强,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第四款；《中华人民共和国刑法（1997年...,广东省东莞市第二市区人民检察院,欧阳科军,男,汉族,1976年12月23日,指控被告人欧阳科军犯贩卖毒品罪、被告人向武强犯容留他人吸毒罪，于2020年12月15日向本院...,有期徒刑三年三个月,NaN,走私、贩卖、运输、制造毒品罪
426,7150,（2021）川0182刑初14号,黄和平、雷茗走私、贩卖、运输、制造毒品罪一审刑事判决书,彭州市人民法院,彭州市,刑事一审,2021-01-15,黄和平；雷茗,走私、贩卖、运输、制造毒品,《中华人民共和国刑法（1997年）》:第三百四十七条第一款；《中华人民共和国刑法（1997年...,四川省彭州市人民检察院,黄和平,男,汉族,1977年9月10日,指控被告人黄和平、雷茗犯贩卖毒品罪，于2021年1月8日向本院提起公诉。本院依法适用速裁程序...,有期徒刑一年二个月,罚金人民币四千元,走私、贩卖、运输、制造毒品罪


In [36]:
result.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品_concat.csv',index=False)

## 实体内容提取

In [12]:
import pandas as pd 
import re
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品_concat.csv')
def extract_text_segment(text):
    # 使用正则提取 "指控" 到 "公诉机关认为" 之间的内容，包括边界词
    match = re.search(r'公诉机关指控.*?公诉机关认为', text)
    return match.group(0) if match else None

# 应用函数到 DataFrame 的 '文本' 列，生成新列 'NER'
df['NER'] = df['文本'].apply(extract_text_segment)

In [14]:
df['NER'].isnull().sum()

295

In [16]:
df.dropna(inplace=True)
len(df)

109

In [17]:
df.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品_final.csv')

In [19]:
import pandas as pd 
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2021.csv')
len(df)

245

In [20]:
import pandas as pd 
df1=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/2021实体关系提取.csv')
df2=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2021.csv')
result = pd.concat([df1, df2], axis=1)
result.columns

Index(['毒品重量', '毒品名称', '地点', '时间', '贩卖(给人)', '携带(毒品)', '持有', '非法容留',
       'Unnamed: 0', 'index', '案号', '案件名称', '法院', '所属地区', '审理程序', '裁判日期',
       '当事人', '案由', '法律依据', '公诉机关', '姓名', '当事人基本情况.性别', '民族', '出生年月', '文本',
       '刑期', '罚金', '罪名', 'NER'],
      dtype='object')

In [21]:
len(result)

245

In [22]:
df=result

# 处理函数：筛选符合 name1 == 被告人 且 name1 != name2 的记录
def filter_relationships(row, column_name):
    value = row[column_name]
    if isinstance(value, str):  # 检查值是否为字符串类型
        relations = value.split(';')  # 多个关系用分号分割
        filtered_relations = []
        for relation in relations:
            parts = relation.strip().split(' - ')  # 拆分为 name1, relation, name2
            if len(parts) == 3:
                name1, _, name2 = parts
                if name1 == row['当事人'] and name1 != name2:
                    filtered_relations.append(relation)  # 满足条件的关系
        return '; '.join(filtered_relations)  # 重新拼接
    return ''  # 如果为空或非字符串，则返回空字符串

# 遍历每一列进行处理
columns_to_check = ['贩卖(给人)', '携带(毒品)', '持有', '非法容留']
for col in columns_to_check:
    df[col] = df.apply(lambda row: filter_relationships(row, col), axis=1)
df

,毒品重量,毒品名称,地点,时间,贩卖(给人),携带(毒品),持有,非法容留,Unnamed: 0,index,...,公诉机关,姓名,当事人基本情况.性别,民族,出生年月,文本,刑期,罚金,罪名,NER
0,[],['冰毒'],"['遵义市播州区南白街道象山社区三小区3栋301室家中', '遵义市播州区']","['2016年9月份的一天', '2016年10月份的一天']",,,,罗昭俊 - 非法容留 - 刘某1; 罗昭俊 - 非法容留 - 刘某2; 罗昭俊 - 非...,3.0,462,...,遵义市播州区人民检察院,罗昭俊,男,汉族,1969年6月16日,指控被告人罗昭俊犯容留他人吸毒罪，向本院提起公诉。本院于2021年1月15日受理后，依法适用...,有期徒刑七个月,罚金人民币一千元,容留他人吸毒罪,公诉机关指控，1、2016年9月份的一天，被告人罗昭俊在遵义市播州区家中，容留刘某1采用吹壶...
1,"['0.69克', '0.94克']","['麻果', '甲基苯丙胺', '甲基苯丙胺片剂', '冰毒']",['武汉市东西湖区将军路办事处将军新村四期幼儿园'],['2020年9月25日'],,何某某 - 携带(毒品) - 甲基苯丙胺片剂; 何某某 - 携带(毒品) - 冰毒; ...,,,6.0,654,...,武汉市东西湖区人民检察院,何某某,男,汉族,1994年5月14日,指控被告人何某某犯贩卖毒品罪。本院适用刑事案件速裁程序，实行独任审判，公开开庭审理了本案。公...,有期徒刑十个月,罚金人民币二千元,走私、贩卖、运输、制造毒品罪,公诉机关指控，2020年9月25日，被告人何某某至武汉市东西湖区将军路办事处将军新村四期幼儿...
2,"['4.83克', '0.63克']","['麻果', '甲基苯丙胺', '甲基苯丙胺片剂', '冰毒']",[],"['2020年11月26日', '2020年11月25日18时10分']",,夏春飞 - 携带(毒品) - 冰毒; 夏春飞 - 携带(毒品) - 甲基苯丙胺; 夏春...,,,7.0,679,...,湖北省武汉市新洲区人民检察院,夏春飞,男,汉族,1983年12月16日,指控被告人夏春飞犯贩卖毒品罪。本院适用刑事案件速裁程序，实行独任审判，公开开庭审理了本案。公...,有期徒刑一年二个月,罚金人民币四千元,走私、贩卖、运输、制造毒品罪,公诉机关指控：2020年11月25日18时10分，被告人夏春飞与刘某电话联系约定以350元的...
3,['0.2克'],['冰毒'],['青岛市市北区太清路45号小区门口'],['2020年8月22日'],,刘丽娜 - 携带(毒品) - 冰毒,,,11.0,760,...,山东省青岛市城阳区人民检察院,刘丽娜,女,汉族,1983年8月30日,指控被告人刘丽娜犯贩卖毒品罪向本院提起公诉。本院受理后，依法组成合议庭，适用一审简易程序（认...,有期徒刑六个月,罚金人民币二千元,走私、贩卖、运输、制造毒品罪,公诉机关指控，2020年8月22日，被告人刘丽娜在位于青岛市市北区太清路45号小区门口将约0...
4,['0.50克'],"['麻果', '“', '甲基苯丙胺片剂']",['武汉市东西湖区某某办事处某某菜场'],['2020年11月12日19时'],,钱某某 - 携带(毒品) - “; 钱某某 - 携带(毒品) - 麻果; 钱某某 - 携...,,,18.0,974,...,武汉市东西湖区人民检察院,钱某某,女,汉族,1975年8月9日,指控被告人钱某某犯贩卖毒品罪。本院适用刑事案件速裁程序，实行独任审判，公开开庭审理了本案。公...,拘役四个月,罚金人民币二千元,走私、贩卖、运输、制造毒品罪,公诉机关指控，2020年11月12日19时许，被告人钱某某在武汉市东西湖区某某办事处某某菜场...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
240,"['0.5克', '0.6克']",['冰毒'],"['青岛市市北区', '青岛市中心医院']","['2020年8月21日22时', '2020年9月中上旬']",,于润涛 - 携带(毒品) - 冰毒,,,NaN,6682,...,山东省青岛市城阳区人民检察院,于润涛,男,汉族,1981年7月11日,指控被告人于润涛犯贩卖毒品罪向本院提起公诉。本院受理后，依法组成合议庭，适用一审简易程序（认...,有期徒刑一年五个月,罚金人民币七千元,走私、贩卖、运输、制造毒品罪,公诉机关指控：1、2020年8月21日22时许，被告人于润涛在青岛市中心医院附近以人民币10...
241,['1克'],['冰毒'],['城阳区'],"['2019年3月17日晚', '2021年1月28日']",,,,,NaN,6685,...,山东省青岛市城阳区人民检察院,吕锡龙,男,汉族,1990年2月21日,指控被告人吕锡龙、王鑫犯贩卖毒品罪向本院提起公诉。本院受理后，依法组成合议庭，适用一审简易程...,有期徒刑七个月,罚金人民币三千元,走私、贩卖、运输、制造毒品罪,公诉机关指控，2019年3月17日晚，被告人吕锡龙收取刘某人民币1400元（以下币种同）购毒...
242,[],"['甲基苯丙胺', '冰毒']","['衡阳市珠晖区', '衡东', '衡东县银都宾馆', '衡东县宾馆', '衡东县宾馆号房'...","['2020年12月13日', '2020年12月16日', '2021年2月20日', '...",,,,,NaN,6730,...,衡阳市蒸湘区人民检察院,尹林,男,汉族,1990年5月14日,指控被告人尹林、戴文凯犯容留他人吸毒罪，向本院提起公诉。本院于2021年6月4日受理后，依法...,拘役五个月,罚金人民币一千元,容留他人吸毒罪,公诉机关指控，被告人戴文凯和被告人尹林自2020年11月份以来，多次互相容留吸毒，其中戴文凯...
243,[],['海洛因'],"['岳阳市马壕原鸡场附近十字路口', '岳阳市路长城加油站']","['22日晚上', '2021年2月20日下午']",,,,,NaN,6773,...,岳阳市岳阳楼区人民检察院,吴健,男,汉族,1971年2月16日,指控被告人吴健、赵本虎犯贩卖毒品罪，向本院提起公诉。本院于2021年6月8日受理后，依法组成...,有期徒刑一年二个月,罚金人民币三千元,走私、贩卖、运输、制造毒品罪,公诉机关指控，2021年2月20日下午，吸毒人员周某联系被告人赵本虎购买200元的毒品海洛因...


In [23]:
df.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2021_final.csv',index=False)

In [24]:
df.columns

Index(['毒品重量', '毒品名称', '地点', '时间', '贩卖(给人)', '携带(毒品)', '持有', '非法容留',
       'Unnamed: 0', 'index', '案号', '案件名称', '法院', '所属地区', '审理程序', '裁判日期',
       '当事人', '案由', '法律依据', '公诉机关', '姓名', '当事人基本情况.性别', '民族', '出生年月', '文本',
       '刑期', '罚金', '罪名', 'NER'],
      dtype='object')

## 切分实体、关系csv

In [1]:
import pandas as pd 
df=pd.read_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2021_final.csv')
df.columns

Index(['毒品重量', '毒品名称', '地点', '时间', '贩卖(给人)', '携带(毒品)', '持有', '非法容留',
       'Unnamed: 0', 'index', '案号', '案件名称', '法院', '所属地区', '审理程序', '裁判日期',
       '被告人', '案由', '法律依据', '公诉机关', '姓名', '性别', '民族', '出生年月', '文本', '刑期', '罚金',
       '罪名', 'NER'],
      dtype='object')

In [2]:
filtered_columns = ['审理程序', '裁判日期', '罪名', '案件名称', '法院', '被告人', '案由', '案号', '刑期', '罚金']
df_entity=df[filtered_columns]
df_entity.columns

Index(['审理程序', '裁判日期', '罪名', '案件名称', '法院', '被告人', '案由', '案号', '刑期', '罚金'], dtype='object')

In [3]:
df_entity.to_csv('/Users/yuu/Downloads/毕业设计/spider_bysj/tmp/毒品2021_entity.csv',index=False)